# SI 313 WN26: Surveys Module Assignment
### Part B: Survey Data Analysis
##### Nick Pisarczyk - 03/25/26
---

## Introduction
- Research question and motivation
- Specific hypotheses you're testing

Choose ONE research question to investigate:

- Trust in institutions: What predicts trust in government institutions (Congress, presidency, media, science)? How do political affiliation, education, and demographic factors relate to institutional trust?
- Information sources and beliefs: How do different news sources (traditional media, social media, partisan outlets) relate to political knowledge, beliefs about factual matters, or policy preferences?
- Identity and political attitudes: How do different social identities (race, gender, education, religion, region) relate to political preferences? Are there interaction effects between identities?

---

## Data Loading and Exploration
- Load ANES data
- Examine variables of interest
- Check missing data patterns
- Create necessary derived variables

In [52]:
# libraries
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

In [53]:
# Load the 2020 ANES Time Series data
# df = pd.read_csv("/Users/libbyh/Documents/git/si313_instructors/answer_keys/data/38034-0001-Data.tsv", sep='\t')
df = pd.read_stata("../data/surveys_module_data/38034-0001-Data.dta", convert_categoricals=False)

# news/media sources
# rename variables with friendly names
df = df.rename(columns={
    "V201629A": "tv_used",                   # data about what media sources people heard anything about the presidential campaign from 
    "V201629B": "newspaper_used",            # covers traditional source use of radio/newspaper
    "V201629C": "internet_used",
    "V201629D": "radio_used",

    "V201630B": "sean_hannity_fox",          # partisan outlets      # watch hannity at least once a month
    "V201630C": "tucker_carlson_fox",                                # ... carlson ...
    "V201630D": "rachel_maddow_msnbc",                               # ... maddow ...
    "V201630E": "lawrence_odonnell_msnbc",                           # ... odonnell ...

    "V201634A": "yahoo_site",                # websites
    "V201634J": "bbc_site",
    "V201634P": "buzzfeed_site",

    "V201635A": "nyt_paper",                 # printed newspaper (traditional sources)
    "V201634B": "usa_today_paper",
    "V201634C": "wsj_paper",
    "V201634D": "wash_post_paper",
})

# political knowledge
# rename variables with friendly names
df = df.rename(columns={
    "V201644": "senator_term",
    "V201645": "least_fed_spending",
    "V201646": "house_majority",
    "V201647": "senate_majority",
})

# factual matters
# rename variables with friendly names
df = df.rename(columns={
    "V202332": "climate_change_effects",     # how much is climate change affecting severe weather/temps in US 
    "V202557": "covid_lab",                  # was covid created in a lab
    "V201333X": "unemployment_attitude",     # was unemployment better or worse than last year
})

# policy preferences
# rename variables with friendly names
df = df.rename(columns={
    "V202342": "ban_assault_rifles",
    "V202374": "12k_federal_aid",
    "V201313": "welfare_programs",
    "V201321": "protect_environment",
})

model_df = df[["tv_used", "newspaper_used", "internet_used", "radio_used",
               "sean_hannity_fox", "tucker_carlson_fox", "rachel_maddow_msnbc", "lawrence_odonnell_msnbc",
               "yahoo_site", "bbc_site", "buzzfeed_site",
               "nyt_paper", "usa_today_paper", "wsj_paper", "wash_post_paper",
               "senator_term", "least_fed_spending", "house_majority", "senate_majority",
               "climate_change_effects", "covid_lab", "unemployment_attitude",
               "ban_assault_rifles", "12k_federal_aid", "welfare_programs", "protect_environment"
               ]].dropna()

print(model_df.head())

   tv_used  newspaper_used  internet_used  radio_used  sean_hannity_fox  \
0        1               0              1           1                 0   
1        1               0              1           0                 0   
2        1               1              1           1                 0   
3        1               0              0           0                 0   
4        1               0              1           1                 0   

   tucker_carlson_fox  rachel_maddow_msnbc  lawrence_odonnell_msnbc  \
0                   0                    0                        0   
1                   0                    0                        0   
2                   0                    0                        0   
3                   1                    0                        0   
4                   0                    0                        1   

   yahoo_site  bbc_site  ...  least_fed_spending  house_majority  \
0           0         0  ...                   3      

In [ ]:
# Recode factual knowledge questions to get binary right or wrong values
# correct answers are 1
# incorrect answers are 0
# all others are NaN

def recode_factual_questions(column_name, correct_val):
    conditions = [
            df[column_name] == correct_val,
            df[column_name] > 0,
            df[column_name] < 0,
        ]
    choices = [
        1,
        0,
        np.nan,
    ]

    model_df[column_name] = np.select(conditions, choices, default=model_df[column_name])
    print(f"Values in column {column_name} are:\n{model_df[column_name].unique()}\n")

In [55]:
recode_factual_questions('senator_term', 6)
recode_factual_questions('least_fed_spending', 1)
recode_factual_questions('house_majority', 1)
recode_factual_questions('senate_majority', 2)

Values in column senator_term are:
[ 1.  0. nan]

Values in column least_fed_spending are:
[ 0.  1. nan]

Values in column house_majority are:
[ 1.  0. nan]

Values in column senate_majority are:
[ 1.  0. nan]



---

## Descriptive analysis
- Summary statistics for key variables
- Crosstabulations or correlations

---

## Statistical Analysis
- Appropriate statistical tests or models for your research question
    - Examples: t-tests, chi-square tests, ANOVA, correlation analysis, regression models
- Interpretation of results with attention to statistical significance AND practical significance
- At least 3 well-labeled visualizations (here or in descriptive analysis)

---

## Discussion of Survey Design
- How might question wording affect your findings?
- What response biases might be present in this data?
- How do ANES sampling and weighting procedures affect interpretation?
- Compare self-report survey data to behavioral/observational data for your topic